In [2]:
"""
Round 4 — Full Narrative Page Reconstruction Pipeline
======================================================
Signals used:
  1. Sentence-transformer boundary embeddings (MiniLM)
  2. GPT-2 transition perplexity
  3. Named-entity overlap (spaCy)
  4. Structural / linguistic features
  5. OR-Tools TSP solver for global chain optimisation

Install requirements:
    pip install pandas numpy scikit-learn sentence-transformers transformers \
                torch spacy ortools tqdm
    python -m spacy download en_core_web_sm
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import re
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import spacy

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp


# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION  — tweak weights here
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # How many chars to use at the boundary of each page
    "boundary_chars": 400,

    # Weight for each signal when building the combined coherence matrix
    "w_embedding":   0.45,   # cosine similarity of page boundaries
    "w_perplexity":  0.35,   # GPT-2 transition naturalness
    "w_entity":      0.10,   # named-entity overlap between adjacent pages
    "w_structural":  0.10,   # linguistic / structural cues

    # OR-Tools time limit per book (seconds)
    "tsp_time_limit": 60,

    # Device for torch models
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

print(f"[config] device = {CFG['device']}")


# ─────────────────────────────────────────────────────────────────────────────
# 2. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
def load_data():
    train  = pd.read_csv("./data/Mysterious_Affair_at_Styles_Train_Data.csv")        # correct-order training book
    book_a = pd.read_csv("./data/BookA_test.csv")   # shuffled test book A
    book_b = pd.read_csv("./data/BookB_test.csv")   # shuffled test book B

    for df in [train, book_a, book_b]:
        df["text"] = df["text"].fillna("").astype(str).str.strip()
        df["text"] = df["text"].apply(lambda t: re.sub(r"\s+", " ", t))

    print(f"[data] train={len(train)} pages | bookA={len(book_a)} | bookB={len(book_b)}")
    return train, book_a, book_b


# ─────────────────────────────────────────────────────────────────────────────
# 3. SIGNAL 1 — SENTENCE-TRANSFORMER BOUNDARY EMBEDDINGS
# ─────────────────────────────────────────────────────────────────────────────
def build_embedding_matrix(texts: list[str], model: SentenceTransformer, n: int) -> np.ndarray:
    """
    Returns an (N x N) matrix where entry [i, j] is the cosine similarity
    between the END of page i and the BEGINNING of page j.
    High similarity  →  page j likely follows page i.
    """
    endings    = [t[-n:] for t in texts]
    beginnings = [t[:n]  for t in texts]

    print("  embedding endings ...")
    emb_end = model.encode(endings,    batch_size=32, show_progress_bar=True,
                           convert_to_numpy=True, normalize_embeddings=True)
    print("  embedding beginnings ...")
    emb_beg = model.encode(beginnings, batch_size=32, show_progress_bar=True,
                           convert_to_numpy=True, normalize_embeddings=True)

    # (N, N) — emb_end[i] · emb_beg[j]
    matrix = emb_end @ emb_beg.T          # cosine sim because both are L2-normalised
    np.fill_diagonal(matrix, -1.0)        # page can't follow itself
    return matrix


# ─────────────────────────────────────────────────────────────────────────────
# 4. SIGNAL 2 — GPT-2 TRANSITION PERPLEXITY
# ─────────────────────────────────────────────────────────────────────────────
def _token_perplexity(text: str, model, tokenizer, device: str, max_len: int = 512) -> float:
    """Lower = more natural / fluent text."""
    enc = tokenizer(text, return_tensors="pt", max_length=max_len,
                    truncation=True).to(device)
    with torch.no_grad():
        loss = model(**enc, labels=enc["input_ids"]).loss
    return loss.item()


def build_perplexity_matrix(texts: list[str], model, tokenizer,
                             device: str, n: int) -> np.ndarray:
    """
    Returns an (N x N) matrix where entry [i, j] reflects how naturally
    page j follows page i (higher = better transition).
    We concatenate the last n chars of page i with the first n chars of page j
    and measure GPT-2 perplexity on the junction region.
    """
    N = len(texts)
    endings    = [t[-n:] for t in texts]
    beginnings = [t[:n]  for t in texts]

    raw = np.zeros((N, N))

    print("  computing perplexity for all pairs (this is the slowest step) ...")
    for i in tqdm(range(N)):
        for j in range(N):
            if i == j:
                raw[i, j] = 999.0
                continue
            junction = endings[i] + " " + beginnings[j]
            raw[i, j] = _token_perplexity(junction, model, tokenizer, device)

    # Convert: lower perplexity → higher score, normalise to [0, 1]
    # We use negative log-perplexity then min-max scale
    neg = -raw
    neg[np.eye(N, dtype=bool)] = np.nan
    lo, hi = np.nanmin(neg), np.nanmax(neg)
    matrix = (neg - lo) / (hi - lo + 1e-9)
    np.fill_diagonal(matrix, 0.0)
    return matrix


# ─────────────────────────────────────────────────────────────────────────────
# 5. SIGNAL 3 — NAMED-ENTITY OVERLAP
# ─────────────────────────────────────────────────────────────────────────────
def build_entity_matrix(texts: list[str], nlp) -> np.ndarray:
    """
    Returns an (N x N) matrix.  Entry [i, j] = Jaccard similarity of named
    entities between pages i and j.  Pages that share many characters / objects
    are likely neighbours, so we use this as a weak proximity signal.
    """
    N = len(texts)
    print("  extracting named entities ...")
    entity_sets = []
    for doc in tqdm(nlp.pipe(texts, batch_size=16, disable=["parser"]), total=N):
        ents = {ent.text.lower() for ent in doc.ents}
        entity_sets.append(ents)

    matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            u = entity_sets[i] | entity_sets[j]
            if not u:
                continue
            matrix[i, j] = len(entity_sets[i] & entity_sets[j]) / len(u)

    return matrix


# ─────────────────────────────────────────────────────────────────────────────
# 6. SIGNAL 4 — STRUCTURAL / LINGUISTIC FEATURES
# ─────────────────────────────────────────────────────────────────────────────
_CONTINUATION_WORDS = {
    "then", "next", "later", "after", "finally", "meanwhile", "suddenly",
    "eventually", "therefore", "thus", "consequently", "however", "but",
    "so", "yet", "still", "nevertheless", "furthermore", "moreover",
    "additionally", "indeed", "also", "besides", "otherwise",
}
_PRONOUNS = {"he", "she", "they", "it", "this", "that", "we", "his", "her",
             "their", "its", "these", "those", "him", "them"}

def _structural_score(end_i: str, begin_j: str) -> float:
    """
    Heuristic score for how likely page j follows page i based on surface cues.
    Returns a value in [0, 1].
    """
    score = 0.0

    # --- Cue 1: j begins with a continuation/pronoun word (max 0.3)
    first_word = begin_j.lower().split()[0] if begin_j.split() else ""
    if first_word in _CONTINUATION_WORDS:
        score += 0.3
    elif first_word in _PRONOUNS:
        score += 0.2

    # --- Cue 2: i ends with a comma, dash, or open-ended conjunction (max 0.2)
    last_char = end_i.rstrip()[-1] if end_i.strip() else ""
    if last_char in {",", ";", "—", "-"}:
        score += 0.2
    elif last_char == ":":
        score += 0.1

    # --- Cue 3: i ends mid-dialogue (open quote not closed) (max 0.15)
    open_quotes  = end_i.count('"') + end_i.count('"')
    close_quotes = end_i.count('"') + end_i.count('"')
    if open_quotes > close_quotes:
        score += 0.15

    # --- Cue 4: j does NOT start a new chapter (penalise if it does) (max 0.1)
    if re.match(r"^(chapter|CHAPTER|Part|PART)\s+[IVXLC\d]", begin_j.strip()):
        score -= 0.25  # chapter boundary → unlikely direct continuation

    # --- Cue 5: word-level unigram overlap at boundary (max 0.25)
    words_end   = set(re.findall(r"\b[a-z]{4,}\b", end_i.lower()))
    words_begin = set(re.findall(r"\b[a-z]{4,}\b", begin_j.lower()))
    if words_end and words_begin:
        overlap = len(words_end & words_begin) / math.sqrt(len(words_end) * len(words_begin))
        score += min(0.25, overlap * 0.5)

    return max(0.0, min(1.0, score))


def build_structural_matrix(texts: list[str], n: int) -> np.ndarray:
    N = len(texts)
    endings    = [t[-n:] for t in texts]
    beginnings = [t[:n]  for t in texts]

    print("  computing structural features ...")
    matrix = np.zeros((N, N))
    for i in tqdm(range(N)):
        for j in range(N):
            if i != j:
                matrix[i, j] = _structural_score(endings[i], beginnings[j])

    return matrix


# ─────────────────────────────────────────────────────────────────────────────
# 7. COMBINE SIGNALS INTO ONE COHERENCE MATRIX
# ─────────────────────────────────────────────────────────────────────────────
def combine_matrices(emb_mat, ppl_mat, ent_mat, struct_mat, cfg: dict) -> np.ndarray:
    """
    Weighted sum of all four normalised signal matrices.
    Each matrix has been independently scaled to [0, 1].
    """
    def _norm(m: np.ndarray) -> np.ndarray:
        lo, hi = m.min(), m.max()
        return (m - lo) / (hi - lo + 1e-9)

    combined = (
        cfg["w_embedding"]  * _norm(emb_mat)    +
        cfg["w_perplexity"] * _norm(ppl_mat)    +
        cfg["w_entity"]     * _norm(ent_mat)    +
        cfg["w_structural"] * _norm(struct_mat)
    )
    np.fill_diagonal(combined, 0.0)
    return combined


# ─────────────────────────────────────────────────────────────────────────────
# 8. GREEDY CHAIN (fast fallback / warm start)
# ─────────────────────────────────────────────────────────────────────────────
def greedy_chain(coherence: np.ndarray) -> list[int]:
    """
    Build a chain greedily: always pick the unused page that scores highest
    as the successor of the current page.

    The starting page is chosen as the one that has the highest "outgoing"
    score relative to its "incoming" score — typical of a first page.
    """
    N = coherence.shape[0]
    out_score = coherence.sum(axis=1)   # total score as a predecessor
    in_score  = coherence.sum(axis=0)   # total score as a successor
    start     = int(np.argmax(out_score - in_score))

    chain = [start]
    used  = {start}

    while len(chain) < N:
        cur = chain[-1]
        row = coherence[cur].copy()
        row[list(used)] = -999.0
        nxt = int(np.argmax(row))
        chain.append(nxt)
        used.add(nxt)

    return chain


# ─────────────────────────────────────────────────────────────────────────────
# 9. OR-TOOLS TSP SOLVER  (global optimum for the chain)
# ─────────────────────────────────────────────────────────────────────────────
def tsp_solve(coherence: np.ndarray, time_limit_sec: int = 60) -> list[int]:
    """
    Treat page ordering as an asymmetric TSP on a complete directed graph.
    cost[i][j] = big_M - coherence[i][j]   (OR-Tools minimises cost)

    Returns a list of page indices in reconstructed order.
    """
    N = coherence.shape[0]
    BIG_M = 10_000

    # OR-Tools requires integer costs
    scale  = 1_000
    cost   = (BIG_M - coherence * scale).astype(int).tolist()

    # Add a dummy depot node (index N) with zero-cost edges so OR-Tools
    # can model an open path (not a closed tour)
    depot_row = [0] * (N + 1)
    for row in cost:
        row.append(0)   # zero cost from any node to depot
    cost.append(depot_row)

    manager = pywrapcp.RoutingIndexManager(N + 1, 1, N)   # 1 vehicle, depot = N
    routing = pywrapcp.RoutingModel(manager)

    def transit(from_idx, to_idx):
        i = manager.IndexToNode(from_idx)
        j = manager.IndexToNode(to_idx)
        return cost[i][j]

    transit_id = routing.RegisterTransitCallback(transit)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_id)

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    params.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    params.time_limit.seconds = time_limit_sec
    params.log_search = False

    solution = routing.SolveWithParameters(params)

    if solution is None:
        print("  [TSP] no solution found — falling back to greedy chain")
        return greedy_chain(coherence)

    # Extract the route (skip depot node N)
    route = []
    idx = routing.Start(0)
    while not routing.IsEnd(idx):
        node = manager.IndexToNode(idx)
        if node != N:
            route.append(node)
        idx = solution.Value(routing.NextVar(idx))

    # Safety: if OR-Tools dropped any node, fall back
    if len(route) != N:
        print("  [TSP] incomplete route — falling back to greedy chain")
        return greedy_chain(coherence)

    print(f"  [TSP] solved  objective = {solution.ObjectiveValue()}")
    return route


# ─────────────────────────────────────────────────────────────────────────────
# 10. POST-PROCESSING — LOCAL 2-OPT REFINEMENT
# ─────────────────────────────────────────────────────────────────────────────
def two_opt(chain: list[int], coherence: np.ndarray, max_passes: int = 5) -> list[int]:
    """
    Try all pairwise swaps.  Accept any swap that improves the total
    adjacent-pair coherence score.  Repeat until no improvement.
    """
    def _score(c):
        return sum(coherence[c[k], c[k + 1]] for k in range(len(c) - 1))

    best = list(chain)
    best_score = _score(best)

    for _pass in range(max_passes):
        improved = False
        for i in range(1, len(best) - 1):
            for j in range(i + 1, len(best)):
                candidate = best[:i] + best[i:j + 1][::-1] + best[j + 1:]
                s = _score(candidate)
                if s > best_score:
                    best       = candidate
                    best_score = s
                    improved   = True
        if not improved:
            break
        print(f"  [2-opt pass {_pass + 1}] score = {best_score:.4f}")

    return best


# ─────────────────────────────────────────────────────────────────────────────
# 11. BUILD COHERENCE MATRIX FOR ONE BOOK
# ─────────────────────────────────────────────────────────────────────────────
def build_full_coherence(texts, embed_model, gpt2_model, gpt2_tok, nlp, cfg):
    n = cfg["boundary_chars"]

    print("\n[signal 1] building embedding matrix ...")
    emb_mat = build_embedding_matrix(texts, embed_model, n)

    print("\n[signal 2] building perplexity matrix ...")
    ppl_mat = build_perplexity_matrix(texts, gpt2_model, gpt2_tok, cfg["device"], n)

    print("\n[signal 3] building entity matrix ...")
    ent_mat = build_entity_matrix(texts, nlp)

    print("\n[signal 4] building structural matrix ...")
    struct_mat = build_structural_matrix(texts, n)

    print("\n[combine] merging signals ...")
    coherence = combine_matrices(emb_mat, ppl_mat, ent_mat, struct_mat, cfg)

    return coherence


# ─────────────────────────────────────────────────────────────────────────────
# 12. FULL RECONSTRUCTION PIPELINE FOR ONE BOOK
# ─────────────────────────────────────────────────────────────────────────────
def reconstruct(book_df: pd.DataFrame, embed_model, gpt2_model, gpt2_tok,
                nlp, cfg: dict, label: str) -> pd.DataFrame:
    print(f"\n{'='*60}")
    print(f"  Reconstructing {label}  ({len(book_df)} pages)")
    print(f"{'='*60}")

    texts = book_df["text"].tolist()

    coherence = build_full_coherence(texts, embed_model, gpt2_model,
                                     gpt2_tok, nlp, cfg)

    print("\n[solve] running OR-Tools TSP solver ...")
    ordered_indices = tsp_solve(coherence, time_limit_sec=cfg["tsp_time_limit"])

    print("\n[refine] 2-opt local search ...")
    ordered_indices = two_opt(ordered_indices, coherence)

    # Map back to original shuffled_page identifiers
    shuffled_pages = book_df.iloc[ordered_indices]["page"].values
    submission = pd.DataFrame({
        "original_page": range(1, len(ordered_indices) + 1),
        "shuffled_page": shuffled_pages,
    })

    fname = f"{label}.csv"
    submission.to_csv(fname, index=False)
    print(f"\n[done] saved → {fname}")
    return submission


# ─────────────────────────────────────────────────────────────────────────────
# 13. OPTIONAL — TRAIN A PAIRWISE CLASSIFIER ON THE TRAINING BOOK
#     (for extra signal; disable if you want a faster run)
# ─────────────────────────────────────────────────────────────────────────────
def train_pairwise_signal(train_df: pd.DataFrame, embed_model,
                          boundary_chars: int = 400) -> np.ndarray:
    """
    Uses the correctly ordered training book to learn what a "good next-page
    boundary" looks like via logistic regression.

    Returns a function  score(emb_end_i, emb_beg_j) → float
    so you can plug it into the main coherence matrix.

    This is optional but squeezes extra accuracy on similarly-styled books.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from itertools import combinations

    pages = train_df.sort_values("page")
    texts = pages["text"].tolist()
    N     = len(texts)

    endings    = [t[-boundary_chars:] for t in texts]
    beginnings = [t[:boundary_chars]  for t in texts]

    print("[pairwise] embedding training book ...")
    emb_end = embed_model.encode(endings,    batch_size=32, show_progress_bar=True,
                                 convert_to_numpy=True, normalize_embeddings=True)
    emb_beg = embed_model.encode(beginnings, batch_size=32, show_progress_bar=True,
                                 convert_to_numpy=True, normalize_embeddings=True)

    X, y = [], []
    for i, j in combinations(range(N), 2):
        # Feature: concatenate [end_i, begin_j, end_i - begin_j, end_i * begin_j]
        fi_j = np.concatenate([emb_end[i], emb_beg[j],
                               emb_end[i] - emb_beg[j],
                               emb_end[i] * emb_beg[j]])
        fj_i = np.concatenate([emb_end[j], emb_beg[i],
                               emb_end[j] - emb_beg[i],
                               emb_end[j] * emb_beg[i]])
        X.extend([fi_j, fj_i])
        # Label: 1 if i < j (i comes before j)
        pi = pages.iloc[i]["page"]
        pj = pages.iloc[j]["page"]
        y.extend([int(pi < pj), int(pj < pi)])

    X = np.array(X)
    y = np.array(y)

    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X)

    clf = LogisticRegression(max_iter=1000, C=1.0)
    clf.fit(X_sc, y)
    print(f"[pairwise] classifier trained  (train accuracy = {clf.score(X_sc, y):.3f})")

    return clf, scaler, emb_end, emb_beg


def apply_pairwise_matrix(book_df, embed_model, clf, scaler,
                          boundary_chars: int = 400) -> np.ndarray:
    """Apply the trained pairwise classifier to get a signal matrix for a test book."""
    texts = book_df["text"].tolist()
    N     = len(texts)

    endings    = [t[-boundary_chars:] for t in texts]
    beginnings = [t[:boundary_chars]  for t in texts]

    emb_end = embed_model.encode(endings,    batch_size=32, show_progress_bar=False,
                                 convert_to_numpy=True, normalize_embeddings=True)
    emb_beg = embed_model.encode(beginnings, batch_size=32, show_progress_bar=False,
                                 convert_to_numpy=True, normalize_embeddings=True)

    matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            feat = np.concatenate([emb_end[i], emb_beg[j],
                                   emb_end[i] - emb_beg[j],
                                   emb_end[i] * emb_beg[j]]).reshape(1, -1)
            feat_sc = scaler.transform(feat)
            matrix[i, j] = clf.predict_proba(feat_sc)[0][1]   # P(i before j)

    return matrix


# ─────────────────────────────────────────────────────────────────────────────
# 14. SCORING UTILITY — evaluate locally (only valid if you know ground truth)
# ─────────────────────────────────────────────────────────────────────────────
def kendall_tau_score(pred_order: list[int], true_order: list[int]) -> float:
    """
    pred_order[i] = which page index (0-based) is at position i
    true_order[i] = ground-truth page index at position i

    Returns normalised Kendall Tau in [0, 1].
    """
    n   = len(pred_order)
    pos = {page: rank for rank, page in enumerate(pred_order)}
    D   = 0
    T   = n * (n - 1) // 2

    for i in range(n):
        for j in range(i + 1, n):
            if pos[true_order[i]] > pos[true_order[j]]:
                D += 1

    tau = 1 - 2 * D / T
    return (tau + 1) / 2


# ─────────────────────────────────────────────────────────────────────────────
# 15. MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    # ----- load data
    train, book_a, book_b = load_data()

    # ----- load models (done once, shared across both books)
    print("\n[models] loading sentence transformer ...")
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")

    print("[models] loading GPT-2 ...")
    gpt2_tok   = GPT2TokenizerFast.from_pretrained("gpt2")
    gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(CFG["device"])
    gpt2_model.eval()
    gpt2_tok.pad_token = gpt2_tok.eos_token

    print("[models] loading spaCy ...")
    nlp = spacy.load("en_core_web_sm")

    # ----- (optional) train pairwise classifier on training book
    #       Adds ~5-10 min but can help when the test book shares the same author
    USE_PAIRWISE = False
    if USE_PAIRWISE:
        clf, scaler, _, _ = train_pairwise_signal(train, embed_model,
                                                  CFG["boundary_chars"])

    # ----- reconstruct Book A
    sub_a = reconstruct(book_a, embed_model, gpt2_model, gpt2_tok,
                        nlp, CFG, label="BookA")

    # ----- reconstruct Book B
    sub_b = reconstruct(book_b, embed_model, gpt2_model, gpt2_tok,
                        nlp, CFG, label="BookB")

    print("\n✓  Both submissions saved as BookA.csv and BookB.csv")
    print("\nSample from BookA:")
    print(sub_a.head(10).to_string(index=False))

    # ----- (optional) self-evaluation on training set
    #       Uncomment only if you have a ground-truth for test books
    # true_order = list(range(len(book_a)))   # replace with actual order
    # pred_order = sub_a["shuffled_page"].tolist()
    # print(f"\nKendall Tau Score: {kendall_tau_score(pred_order, true_order):.4f}")


if __name__ == "__main__":
    main()

[config] device = cuda
[data] train=207 pages | bookA=147 | bookB=57

[models] loading sentence transformer ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[models] loading GPT-2 ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...


[signal 3] building entity matrix ...
  extracting named entities ...


100%|██████████| 57/57 [00:02<00:00, 26.15it/s]



[signal 4] building structural matrix ...
  computing structural features ...


100%|██████████| 57/57 [00:00<00:00, 368.94it/s]



[combine] merging signals ...

[solve] running OR-Tools TSP solver ...
  [TSP] solved  objective = 524513

[refine] 2-opt local search ...

[done] saved → BookB.csv

✓  Both submissions saved as BookA.csv and BookB.csv

Sample from BookA:
 original_page  shuffled_page
             1             50
             2             76
             3             54
             4            104
             5             41
             6             21
             7            145
             8             63
             9              5
            10            111
